# LeetCode 1317 — Convert Integer to the Sum of Two No-Zero Integers bằng Python

**Link đề:** https://leetcode.com/problems/convert-integer-to-the-sum-of-two-no-zero-integers/
**Độ khó:** Easy · **Chủ đề:** Math (xử lý chữ số, thử lần lượt, xây dựng theo từng chữ số)

```python
class Solution:
    def getNoZeroIntegers(self, n: int) -> list[int]:
        ...
```

## Mục tiêu học
1. Hiểu khái niệm **No-Zero integer** và cách kiểm tra "một số có chứa chữ số 0 không" bằng **chuỗi** và bằng **toán** (`% 10`, `// 10`).
2. Viết lời giải **thử lần lượt** (brute force) và biết vì sao nó đủ nhanh với constraints.
3. Học lời giải **xây dựng theo từng chữ số** có "mượn" (borrow) như phép trừ tay — `O(số chữ số)`.
4. Viết **validator** cho bài nhiều đáp án, trace, chứng minh, Time/Space.
5. Luyện biến thể: chia thành 3 số, cấm chữ số khác, đếm số cặp.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

**No-Zero integer** là số nguyên **dương** mà biểu diễn thập phân **không chứa chữ số `0`**.

Cho `n`, trả về list `[a, b]` sao cho:
- `a` và `b` đều là No-Zero integer,
- `a + b = n`.

Đề đảm bảo luôn có ít nhất một đáp án. Có nhiều đáp án thì trả về **bất kỳ** cái nào.

| `n` | Output mẫu | Đáp án khác cũng đúng |
|---:|---|---|
| `2` | `[1, 1]` | — |
| `11` | `[2, 9]` | `[8, 3]`, `[9, 2]`... (nhưng **không** được `[1, 10]` vì 10 có chữ số 0) |

**Constraints:** `2 <= n <= 10^4`.

Câu hỏi trước khi code:
- Vì sao `[1, n - 1]` không phải lúc nào cũng đúng? Thử `n = 11`, `n = 101`.
- Có bao nhiêu cặp `(a, b)` cần thử trong trường hợp xấu nhất?

## 2. Nền tảng Python

### 2.1. Kiểm tra chữ số 0 bằng chuỗi: `'0' in str(x)`

- `str(105)` → `"105"`.
- Toán tử `in` với chuỗi kiểm tra **chuỗi con**: `'0' in "105"` → `True`.
- `not in` là phủ định: `'0' not in "123"` → `True`.

In [1]:
for x in [105, 123, 10, 9, 1000]:
    print(x, "-> str:", repr(str(x)), "| có '0'?", '0' in str(x), "| No-Zero?", '0' not in str(x))

105 -> str: '105' | có '0'? True | No-Zero? False
123 -> str: '123' | có '0'? False | No-Zero? True
10 -> str: '10' | có '0'? True | No-Zero? False
9 -> str: '9' | có '0'? False | No-Zero? True
1000 -> str: '1000' | có '0'? True | No-Zero? False


### 2.2. Kiểm tra chữ số 0 bằng toán: `% 10` và `// 10`

- `x % 10` lấy **chữ số cuối**: `105 % 10 = 5`.
- `x // 10` **bỏ chữ số cuối**: `105 // 10 = 10`.

Lặp lại cho đến khi `x == 0`, ta đi qua từng chữ số từ phải sang trái.

In [2]:
x = 105
while x > 0:
    print(f"x = {x:<4} chữ số cuối x % 10 = {x % 10}, bỏ đi -> x // 10 = {x // 10}")
    x //= 10

x = 105  chữ số cuối x % 10 = 5, bỏ đi -> x // 10 = 10
x = 10   chữ số cuối x % 10 = 0, bỏ đi -> x // 10 = 1
x = 1    chữ số cuối x % 10 = 1, bỏ đi -> x // 10 = 0


In [3]:
def is_no_zero(x: int) -> bool:
    if x <= 0:                 # No-Zero phải là số dương
        return False
    while x > 0:
        if x % 10 == 0:
            return False
        x //= 10
    return True

for x in [105, 123, 10, 9, 1000, 0]:
    print(x, is_no_zero(x), "| so với cách chuỗi:", x > 0 and '0' not in str(x))

105 False | so với cách chuỗi: False
123 True | so với cách chuỗi: True
10 False | so với cách chuỗi: False
9 True | so với cách chuỗi: True
1000 False | so với cách chuỗi: False
0 False | so với cách chuỗi: False


### 2.3. Validator cho bài "trả về bất kỳ"

Vì có nhiều đáp án đúng, ta kiểm tra bằng **điều kiện của đề**, không so `==` với output mẫu.

In [4]:
def is_valid(pair: list[int], n: int) -> bool:
    return len(pair) == 2 and is_no_zero(pair[0]) and is_no_zero(pair[1]) and pair[0] + pair[1] == n

print(is_valid([2, 9], 11), is_valid([8, 3], 11))   # True True
print(is_valid([1, 10], 11))                        # False: 10 chứa 0
print(is_valid([0, 11], 11))                        # False: 0 không dương

True True
False
False


## 3. Hiểu bài bằng tay — `[1, n-1]` hỏng khi nào?

Thử `a = 1`, `b = n - 1`:

| n | b = n - 1 | b có 0? | Ổn? |
|---|---|---|---|
| 5 | 4 | không | ✔ |
| 11 | 10 | **có** | ✘ → thử a = 2: b = 9 ✔ |
| 101 | 100 | **có** | ✘ → a = 2: b = 99 ✔ |
| 110 | 109 | **có** | ✘ → a = 2: b = 108 ✘ → ... |

Vậy chỉ cần **thử tiếp** `a = 2, 3, ...` cho đến khi cả hai đều không có 0. Câu hỏi: phải thử bao nhiêu lần?

In [5]:
for n in [5, 11, 101, 110, 1010]:
    a = 1
    while not (is_no_zero(a) and is_no_zero(n - a)):
        a += 1
    print(f"n={n}: cần thử đến a={a}, b={n - a}")

n=5: cần thử đến a=1, b=4
n=11: cần thử đến a=2, b=9
n=101: cần thử đến a=2, b=99
n=110: cần thử đến a=11, b=99
n=1010: cần thử đến a=11, b=999


## 4. Cách trực giác / baseline — thử lần lượt `a = 1, 2, ...`

Đây chính là hint của đề: duyệt `a` từ `1` đến `n - 1`, đặt `b = n - a`, trả về ngay cặp đầu tiên hợp lệ.

In [6]:
def no_zero_brute_string(n: int) -> list[int]:
    for a in range(1, n):
        b = n - a
        if '0' not in str(a) and '0' not in str(b):
            return [a, b]
    return []   # không xảy ra vì đề đảm bảo có đáp án

for n in [2, 11, 19, 139, 1010, 10000]:
    print(n, "->", no_zero_brute_string(n))

2 -> [1, 1]
11 -> [2, 9]
19 -> [1, 18]
139 -> [1, 138]
1010 -> [11, 999]
10000 -> [1, 9999]


### 4.1. Brute force có đủ nhanh không?

Xấu nhất phải thử bao nhiêu `a`? Ta đo trên **toàn bộ** miền `2..10^4`.

In [7]:
def tries_needed(n):
    a = 1
    while not (is_no_zero(a) and is_no_zero(n - a)):
        a += 1
    return a

worst = max(range(2, 10001), key=tries_needed)
print("n khó nhất:", worst, "| số lần thử:", tries_needed(worst))

n khó nhất: 1101 | số lần thử: 112


Trường hợp xấu nhất (`n = 1101`) cũng chỉ hơn 100 lần thử — rất nhỏ so với `n`. Mỗi lần kiểm tra tốn `O(d)` với `d` = số chữ số (≤ 5). Nên brute force chạy rất nhanh với `n <= 10^4`.

Trên lý thuyết, giới hạn trên đơn giản là `O(n · d)` (nếu phải thử tới `n - 1`), nhưng thực tế số lần thử nhỏ hơn rất nhiều. Đây là lý do LeetCode xếp bài Easy.

## 5. Nút thắt và insight — xây từng chữ số như phép trừ tay

Nút thắt: brute force "may mắn" nhanh, nhưng ta không **chứng minh** được số lần thử nhỏ một cách dễ dàng. Có cách nào **xây thẳng** đáp án?

**Insight:** xét `n` từng chữ số từ phải sang trái. Với chữ số `d`:
- Nếu `d >= 2`: tách `d = 1 + (d - 1)` → `a` nhận `1`, `b` nhận `d - 1` (đều khác 0).
- Nếu `d` là `0` hoặc `1` (và còn chữ số phía trước): **mượn 1** từ hàng trên, biến thành `d + 10`, tách `= (d + 1) + 9` → `a` nhận `d + 1` (1 hoặc 2), `b` nhận `9`.
- Nếu là chữ số **đầu tiên** (không còn hàng trên) và `d == 1`: `a` nhận `1`, `b` nhận `0` — nhưng đây là **số 0 đứng đầu** của `b` nên không được viết ra, không vi phạm.

Ví dụ `n = 100` (làm như phép trừ tay):
```text
hàng đơn vị: d = 0, còn hàng trên -> mượn: 10 = 1 + 9   ; phần còn lại 10 - 1 = 9
hàng chục  : d = 9 (chính là số 10 sau khi bị mượn) -> 9 = 1 + 8 ; phần còn lại 0
=> a = 11, b = 89, 11 + 89 = 100
```
Kỹ thuật: tách chữ số cuối bằng `d = n % 10`, phần còn lại `n // 10`; khi mượn thì trừ `1` vào phần còn lại.

In [8]:
# Mô phỏng phép "mượn" cho n = 100, chỉ in d và phần còn lại
rest = 100
while rest > 0:
    d, rest = rest % 10, rest // 10
    if d <= 1 and rest > 0:
        rest -= 1
        print(f"d={d} -> mượn: tách {d + 10} = {d + 1} + 9, phần còn lại sau mượn = {rest}")
    else:
        print(f"d={d} -> tách {d} = 1 + {d - 1}, phần còn lại = {rest}")

d=0 -> mượn: tách 10 = 1 + 9, phần còn lại sau mượn = 9
d=9 -> tách 9 = 1 + 8, phần còn lại = 0


## 6. Lời giải chính — xây theo chữ số, giảng từng đoạn

**Đoạn 1:** `a = b = 0`, `step = 1` (giá trị hàng hiện tại: 1, 10, 100, ...).

**Đoạn 2:** `while n > 0`: lấy `d = n % 10`, `n //= 10`.

**Đoạn 3:**
- nếu `d >= 2` **hoặc** `n == 0` (đây là chữ số đầu): `a += 1*step`, `b += (d-1)*step`.
- ngược lại (`d` là 0/1, còn hàng trên): `a += (d+1)*step`, `b += 9*step`, **mượn**: `n -= 1`.

**Đoạn 4:** `step *= 10`, lặp. Cuối cùng trả `[a, b]`.

Vì sao trường hợp "chữ số đầu" với `d == 1` an toàn? Khi đó `b` nhận `0` ở hàng **cao nhất** — tức không viết ra. Và `b` vẫn dương vì các hàng thấp hơn đã có chữ số ≥ 1 (với `n >= 2`, nếu `n` có 1 chữ số thì `d >= 2`).

In [9]:
def no_zero_digits(n: int) -> list[int]:
    a = b = 0
    step = 1
    while n > 0:
        d = n % 10
        n //= 10
        if d >= 2 or n == 0:
            a += 1 * step
            b += (d - 1) * step
        else:                       # d là 0 hoặc 1, còn hàng phía trên -> mượn
            a += (d + 1) * step
            b += 9 * step
            n -= 1
        step *= 10
    return [a, b]

for n in [2, 11, 19, 100, 139, 1010, 10000]:
    pair = no_zero_digits(n)
    print(n, "->", pair, "hợp lệ?", is_valid(pair, n))

2 -> [1, 1] hợp lệ? True
11 -> [2, 9] hợp lệ? True
19 -> [11, 8] hợp lệ? True
100 -> [11, 89] hợp lệ? True
139 -> [111, 28] hợp lệ? True
1010 -> [111, 899] hợp lệ? True
10000 -> [1111, 8889] hợp lệ? True


## 7. TRACE từng bước

Trace `n = 1010`:

| bước | d | n sau `//10` | nhánh | a | b | n sau mượn | step |
|---|---|---|---|---|---|---|---|
| 1 | 0 | 101 | mượn | 1 | 9 | 100 | 1 |
| 2 | 0 | 10 | mượn | 11 | 99 | 9 | 10 |
| 3 | 9 | 0 | thường (d≥2) | 111 | 899 | 0 | 100 |

Kết quả `[111, 899]`, `111 + 899 = 1010` ✔.

In [10]:
def trace_no_zero_digits(n: int) -> list[int]:
    print(f"n = {n}")
    a = b = 0
    step = 1
    original = n
    while n > 0:
        d = n % 10
        n //= 10
        if d >= 2 or n == 0:
            a += step; b += (d - 1) * step
            branch = "thường"
        else:
            a += (d + 1) * step; b += 9 * step; n -= 1
            branch = "mượn 1"
        print(f"  step={step:<5} d={d}  nhánh={branch:<7} a={a:<6} b={b:<6} phần còn lại n={n}")
        step *= 10
    print(f"  => [{a}, {b}], tổng = {a + b} (== {original}? {a + b == original})")
    return [a, b]

trace_no_zero_digits(1010)
print()
trace_no_zero_digits(11)
print()
trace_no_zero_digits(139)

n = 1010
  step=1     d=0  nhánh=mượn 1  a=1      b=9      phần còn lại n=100
  step=10    d=0  nhánh=mượn 1  a=11     b=99     phần còn lại n=9
  step=100   d=9  nhánh=thường  a=111    b=899    phần còn lại n=0
  => [111, 899], tổng = 1010 (== 1010? True)

n = 11
  step=1     d=1  nhánh=mượn 1  a=2      b=9      phần còn lại n=0
  => [2, 9], tổng = 11 (== 11? True)

n = 139
  step=1     d=9  nhánh=thường  a=1      b=8      phần còn lại n=13
  step=10    d=3  nhánh=thường  a=11     b=28     phần còn lại n=1
  step=100   d=1  nhánh=thường  a=111    b=28     phần còn lại n=0
  => [111, 28], tổng = 139 (== 139? True)


[111, 28]

Chú ý ở `n = 11`: bước 1 `d = 1` còn hàng trên → mượn, `a = 2`, `b = 9`, phần còn lại `1 - 1 = 0` → dừng. Kết quả `[2, 9]` — trùng output mẫu của đề.

## 8. Code nộp LeetCode

### Bản brute force (đơn giản, đủ nhanh — nên dùng khi thi)

In [11]:
class Solution:
    def getNoZeroIntegers(self, n: int) -> list[int]:
        for a in range(1, n):
            b = n - a
            if '0' not in str(a) and '0' not in str(b):
                return [a, b]
        return []

for n in [2, 11, 19, 139]:
    print(n, Solution().getNoZeroIntegers(n))

2 [1, 1]
11 [2, 9]
19 [1, 18]
139 [1, 138]


### Bản xây theo chữ số (`O(số chữ số)`)

In [12]:
class SolutionDigits:
    def getNoZeroIntegers(self, n: int) -> list[int]:
        a = b = 0
        step = 1
        while n > 0:
            d = n % 10
            n //= 10
            if d >= 2 or n == 0:
                a += step
                b += (d - 1) * step
            else:
                a += (d + 1) * step
                b += 9 * step
                n -= 1
            step *= 10
        return [a, b]

for n in [2, 11, 19, 139]:
    print(n, SolutionDigits().getNoZeroIntegers(n))

2 [1, 1]
11 [2, 9]
19 [11, 8]
139 [111, 28]


### Cách 3 — brute force với kiểm tra bằng toán (không dùng chuỗi)

Giống cách 1 nhưng thay `'0' in str(x)` bằng hàm `is_no_zero` dùng `% 10`. Hữu ích khi phỏng vấn yêu cầu "không đổi sang chuỗi".

In [13]:
class SolutionBruteMath:
    def getNoZeroIntegers(self, n: int) -> list[int]:
        def ok(x: int) -> bool:          # hàm lồng: định nghĩa hàm bên trong hàm
            while x:
                if x % 10 == 0:
                    return False
                x //= 10
            return True
        for a in range(1, n):
            if ok(a) and ok(n - a):
                return [a, n - a]
        return []

for n in [2, 11, 19, 139]:
    print(n, SolutionBruteMath().getNoZeroIntegers(n))

2 [1, 1]
11 [2, 9]
19 [1, 18]
139 [1, 138]


`while x:` nghĩa là "khi `x` khác 0" — số `0` được coi là **sai** (falsy) trong điều kiện.

### Bảng so sánh

| Cách | Ý tưởng | Time | Space | Ghi chú |
|---|---|---|---|---|
| 1. Brute + chuỗi | thử `a = 1, 2, ...`, kiểm `'0' in str` | xấu nhất `O(n·d)`, thực tế rất ít lần thử | `O(d)` | dễ viết nhất |
| 2. Brute + toán | như trên, kiểm bằng `% 10` | như trên | `O(1)` | không dùng chuỗi |
| 3. Xây theo chữ số | tách từng chữ số, có mượn | `O(d)` | `O(1)` | chứng minh được, nhanh nhất |

`d` = số chữ số của `n` (≤ 5 với `n <= 10^4`).

## 9. Vì sao cách xây theo chữ số đúng?

**Invariant:** sau mỗi vòng, `a + b + n_còn_lại * step == n_ban_đầu`, và các chữ số đã ghi vào `a`, `b` đều khác 0 (trừ chữ số `0` đứng đầu của `b` ở bước cuối).

- Nhánh thường: hàng hiện tại có giá trị `d·step`, ta ghi `1·step + (d-1)·step = d·step` ✔.
- Nhánh mượn: lấy thêm `10·step` từ hàng trên (bằng `n -= 1` ở phần còn lại, vì phần còn lại được nhân `step*10`), ghi `(d+1)·step + 9·step = (d+10)·step` ✔.
- Chữ số ghi vào `a`: `1` hoặc `d+1 ∈ {1, 2}` — luôn khác 0. Vào `b`: `d-1 ≥ 1` (khi `d ≥ 2`) hoặc `9`. Ngoại lệ duy nhất: chữ số đầu `d = 1` cho `b` số `0` ở hàng cao nhất = không được viết.
- Khi mượn, phần còn lại có thể thành `0` (ví dụ `n = 10`): vòng lặp dừng, không có hàng trên nào bị bỏ sót.

Kiểm chứng toàn miền `2..10^4` ở phần test. ∎

### 9.1. Độ phức tạp

- **Brute force:** mỗi lần thử `O(d)`; số lần thử xấu nhất đo được rất nhỏ (xem 4.1) — cận trên an toàn `O(n·d)`. Space `O(d)` (chuỗi tạm) hoặc `O(1)`.
- **Xây theo chữ số:** `O(d)` = `O(log10 n)` thời gian, `O(1)` bộ nhớ.

## 10. Bộ test

Dùng validator `is_valid`. Test mẫu đề, biên (`n = 2`, `n = 10`, `n = 10^4`), các số "nhiều 0" (`100`, `1000`, `1010`, `1001`), và kiểm tra **toàn bộ miền** cho mọi cách.

In [14]:
def check(name, pair, n):
    assert is_valid(pair, n), (name, n, pair)
    print(f"{name:<18} n={n:<6} -> {pair} PASS")

for cls in [Solution, SolutionDigits, SolutionBruteMath]:
    s = cls()
    print("==", cls.__name__)
    check("ví dụ 1", s.getNoZeroIntegers(2), 2)
    check("ví dụ 2", s.getNoZeroIntegers(11), 11)
    check("testcase 19", s.getNoZeroIntegers(19), 19)
    check("testcase 139", s.getNoZeroIntegers(139), 139)
    check("10 (có 0)", s.getNoZeroIntegers(10), 10)
    check("lớn nhất", s.getNoZeroIntegers(10000), 10000)

== Solution
ví dụ 1            n=2      -> [1, 1] PASS
ví dụ 2            n=11     -> [2, 9] PASS
testcase 19        n=19     -> [1, 18] PASS
testcase 139       n=139    -> [1, 138] PASS
10 (có 0)          n=10     -> [1, 9] PASS
lớn nhất           n=10000  -> [1, 9999] PASS
== SolutionDigits
ví dụ 1            n=2      -> [1, 1] PASS
ví dụ 2            n=11     -> [2, 9] PASS
testcase 19        n=19     -> [11, 8] PASS
testcase 139       n=139    -> [111, 28] PASS
10 (có 0)          n=10     -> [1, 9] PASS
lớn nhất           n=10000  -> [1111, 8889] PASS
== SolutionBruteMath
ví dụ 1            n=2      -> [1, 1] PASS
ví dụ 2            n=11     -> [2, 9] PASS
testcase 19        n=19     -> [1, 18] PASS
testcase 139       n=139    -> [1, 138] PASS
10 (có 0)          n=10     -> [1, 9] PASS
lớn nhất           n=10000  -> [1, 9999] PASS


In [15]:
for f in [no_zero_brute_string, no_zero_digits,
          Solution().getNoZeroIntegers, SolutionDigits().getNoZeroIntegers, SolutionBruteMath().getNoZeroIntegers]:
    for n in range(2, 10001):
        assert is_valid(f(n), n), (f, n)
print("Mọi cách đều hợp lệ trên toàn bộ n = 2..10000")

Mọi cách đều hợp lệ trên toàn bộ n = 2..10000


In [16]:
# Phản ví dụ cho validator
for pair, n, why in [([1, 10], 11, "10 chứa 0"), ([0, 5], 5, "0 không dương"), ([3, 3], 7, "tổng sai")]:
    assert not is_valid(pair, n)
    print("validator bắt được:", why)

validator bắt được: 10 chứa 0
validator bắt được: 0 không dương
validator bắt được: tổng sai


## 11. Lỗi người mới hay gặp

### Lỗi 1: Trả luôn `[1, n - 1]`

In [17]:
def wrong_one(n):
    return [1, n - 1]

for n in [5, 11, 101]:
    print(n, wrong_one(n), "hợp lệ?", is_valid(wrong_one(n), n))

5 [1, 4] hợp lệ? True
11 [1, 10] hợp lệ? False
101 [1, 100] hợp lệ? False


### Lỗi 2: Chỉ kiểm tra `a`, quên `b`

```python
if '0' not in str(a):      # thiếu kiểm tra b
    return [a, n - a]
```

### Lỗi 3: Kiểm tra bằng toán nhưng xử lý `x = 0` sai

Hàm `while x > 0: ...; return True` trả `True` cho `x = 0` (vòng lặp không chạy). Ở brute force `a, b >= 1` nên không sao, nhưng nếu tái sử dụng hàm ở chỗ khác cần nhớ No-Zero **phải dương**.

### Lỗi 4: Duyệt `range(0, n)` — bắt đầu từ `a = 0`

In [18]:
def wrong_only_a(n):
    for a in range(1, n):
        if '0' not in str(a):
            return [a, n - a]

def wrong_start_zero(n):
    for a in range(0, n):
        b = n - a
        if '0' not in str(b) and '0' not in str(a) or a == 0:   # ai đó "quên" xử lý a = 0
            return [a, b]

print("chỉ kiểm a  :", wrong_only_a(11), is_valid(wrong_only_a(11), 11))
print("bắt đầu từ 0:", wrong_start_zero(11), is_valid(wrong_start_zero(11), 11))

chỉ kiểm a  : [1, 10] False
bắt đầu từ 0: [0, 11] False


### Lỗi 5 (cách chữ số): quên `n -= 1` khi mượn

Khi đó hàng trên bị tính thừa `1` → tổng `a + b` lớn hơn `n` đúng `10·step`.

In [19]:
def wrong_no_borrow(n):
    a = b = 0; step = 1
    while n > 0:
        d = n % 10; n //= 10
        if d >= 2 or n == 0:
            a += step; b += (d - 1) * step
        else:
            a += (d + 1) * step; b += 9 * step     # quên n -= 1
        step *= 10
    return [a, b]

print(wrong_no_borrow(11), "tổng =", sum(wrong_no_borrow(11)), "(cần 11)")

[12, 9] tổng = 21 (cần 11)


## 12. Mẹo nhận dạng pattern

- "Số không chứa chữ số X" ⇒ `str(x)` + `in`, hoặc vòng `% 10` / `// 10`.
- Đề "return any" + constraints nhỏ ⇒ **thử lần lượt** thường đủ; luôn ước lượng số lần thử.
- Muốn lời giải chứng minh được ⇒ nghĩ theo **từng chữ số** như phép cộng/trừ tay (có nhớ / có mượn).
- Đáp án nhiều ⇒ test bằng validator, không so với output mẫu.

## 13. Biến thể

1. **Cấm chữ số khác** (ví dụ cấm `7`): brute force chỉ cần đổi `'0'` thành `'7'`.
2. **Tách thành 3 số No-Zero**: thử `a`, rồi giải bài 2 số cho `n - a`.
3. **Đếm** số cặp `(a, b)` với `a <= b` thoả mãn.

In [20]:
def split_avoid_digit(n: int, bad: str) -> list[int]:
    for a in range(1, n):
        if bad not in str(a) and bad not in str(n - a):
            return [a, n - a]
    return []

print("cấm '7', n=14:", split_avoid_digit(14, '7'))
print("cấm '1', n=20:", split_avoid_digit(20, '1'), "<- không tồn tại cặp nào; vì thế đề phải 'đảm bảo có đáp án'")

cấm '7', n=14: [1, 13]
cấm '1', n=20: [] <- không tồn tại cặp nào; vì thế đề phải 'đảm bảo có đáp án'


## 14. Bài tập tự luyện (TODO — tăng dần)

**Bài 1 (tính tay):** Dùng cách xây theo chữ số cho `n = 2020`. Lập bảng `d`, nhánh, `a`, `b`.

**Bài 2 (đoán output):** `Solution().getNoZeroIntegers(100)` (brute force chuỗi) trả về gì?

**Bài 3 (tự viết):** `exercise_is_no_zero(x)` bằng toán, `x <= 0` trả `False`.

**Bài 4 (tự viết):** `exercise_brute(n)` — brute force dùng `exercise_is_no_zero`.

**Bài 5 (biến thể):** `exercise_split_three(n)` → `[a, b, c]` đều No-Zero, tổng `n` (`n >= 3`).

**Bài 6 (mini challenge):** `exercise_count_pairs(n)` đếm số cặp `a <= b`, `a + b = n`, cả hai No-Zero.

In [21]:
def exercise_is_no_zero(x: int) -> bool:
    # TODO
    pass

def exercise_brute(n: int) -> list[int]:
    # TODO
    pass

# assert exercise_is_no_zero(123) and not exercise_is_no_zero(105) and not exercise_is_no_zero(0)
# assert is_valid(exercise_brute(11), 11)

In [22]:
def exercise_split_three(n: int) -> list[int]:
    # TODO
    pass

def exercise_count_pairs(n: int) -> int:
    # TODO
    pass

# for n in range(3, 500):
#     t = exercise_split_three(n)
#     assert len(t) == 3 and sum(t) == n and all(exercise_is_no_zero(x) for x in t)
# assert exercise_count_pairs(11) == 4      # (2,9) (3,8) (4,7) (5,6)

## 15. Gợi ý 3 mức

**Bài 1** — (1) Đi từ hàng đơn vị. (2) `d = 0` còn hàng trên → mượn. (3) Kết quả tổng phải là 2020; kiểm bằng `trace_no_zero_digits(2020)`.

**Bài 2** — (1) `a = 1` → `b = 99`? (2) `99` có chữ số 0 không? (3) Có ngay ở lần thử đầu.

**Bài 3** — (1) Xử lý `x <= 0` trước. (2) Vòng `while x > 0` với `x % 10`. (3) Gặp `0` thì `return False`.

**Bài 5** — (1) Chọn `a` No-Zero. (2) `n - a` phải tách được thành 2 số No-Zero (≥ 2). (3) Dùng lại `no_zero_brute_string(n - a)`, nếu ra `[]` thì thử `a` khác.

**Bài 6** — (1) `a` chạy từ `1` đến `n // 2`. (2) Kiểm cả `a` và `n - a`. (3) Đếm bằng biến `count`.

## 16. Đáp án tham khảo

**Bài 1:** `n = 2020` → `[1121, 899]` (hàng đơn vị `0` mượn → `1+9`; hàng chục `1` mượn → `2+9`; hàng trăm `0` đã thành `9` sau khi bị mượn → `1+8`; hàng nghìn `1` là chữ số đầu → `1+0`). Cell dưới in trace để đối chiếu.

**Bài 2:** `[1, 99]`.

In [23]:
pair_2020 = trace_no_zero_digits(2020)

def answer_is_no_zero(x):
    if x <= 0:
        return False
    while x > 0:
        if x % 10 == 0:
            return False
        x //= 10
    return True

def answer_brute(n):
    for a in range(1, n):
        if answer_is_no_zero(a) and answer_is_no_zero(n - a):
            return [a, n - a]
    return []

def answer_split_three(n):
    for a in range(1, n - 1):
        if answer_is_no_zero(a):
            rest = answer_brute(n - a)
            if rest:
                return [a] + rest
    return []

def answer_count_pairs(n):
    count = 0
    for a in range(1, n // 2 + 1):
        if answer_is_no_zero(a) and answer_is_no_zero(n - a):
            count += 1
    return count

assert pair_2020 == [1121, 899] and is_valid(pair_2020, 2020)
assert Solution().getNoZeroIntegers(100) == [1, 99]
assert answer_is_no_zero(123) and not answer_is_no_zero(105) and not answer_is_no_zero(0)
assert is_valid(answer_brute(11), 11)
for n in range(3, 500):
    t = answer_split_three(n)
    assert len(t) == 3 and sum(t) == n and all(answer_is_no_zero(x) for x in t), n
assert answer_count_pairs(11) == 4
print("Tất cả đáp án PASS!")

n = 2020
  step=1     d=0  nhánh=mượn 1  a=1      b=9      phần còn lại n=201
  step=10    d=1  nhánh=mượn 1  a=21     b=99     phần còn lại n=19
  step=100   d=9  nhánh=thường  a=121    b=899    phần còn lại n=1
  step=1000  d=1  nhánh=thường  a=1121   b=899    phần còn lại n=0
  => [1121, 899], tổng = 2020 (== 2020? True)
Tất cả đáp án PASS!


## 17. Template nhớ nhanh + checklist submit

```python
class Solution:
    def getNoZeroIntegers(self, n: int) -> list[int]:
        for a in range(1, n):                         # a bắt đầu từ 1
            b = n - a
            if '0' not in str(a) and '0' not in str(b):   # kiểm CẢ HAI
                return [a, b]
```

Cách chứng minh được: tách theo chữ số, `d >= 2` → `1 + (d-1)`; `d ∈ {0,1}` → mượn, `(d+1) + 9`.

**Checklist:**
- [ ] `a` bắt đầu từ `1`, không phải `0`.
- [ ] Kiểm tra **cả** `a` và `b`.
- [ ] Đã thử `n = 11`, `n = 100`, `n = 10000`.
- [ ] Test bằng validator, không so với output mẫu `[2, 9]`.